# 实战项目1：训练生成式问答模型

In [52]:
import random
import torch
import numpy as np
import os
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader, random_split
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, get_scheduler
from tqdm.auto import tqdm
import json, sacrebleu
from torch.utils.tensorboard import SummaryWriter
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction


In [14]:
# args:
dataset_size = 15_000
max_input_length = 512
max_target_length = 128

batch_size = 8
learning_rate = 2e-5
epoch_num = 1
log_step = 20
writer = SummaryWriter(log_dir='../tf-logs/t5QA/runs')

In [12]:
class QADataset(Dataset):
    def __init__(self, data_file):
        self.data = self.load_data(data_file)
    
    def load_data(self, data_file):
        Data = {}
        with open(data_file, 'rt', encoding='utf-8') as f:
            for idx, line in enumerate(f):
                if idx >= dataset_size:
                    break
                sample = json.loads(line.strip())
                Data[idx] = sample
        return Data
    
    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

In [4]:
data = QADataset("./dataset/DuReaderQG/train.json")
train_size = int(len(data)*0.9)
valid_size = len(data) - train_size
train_data, valid_data = random_split(data, [train_size, valid_size])

print(f"训练数据集：{train_size}\n验证数据集{valid_size}")
print(f"取出一个样本{train_data[0]}")

训练数据集：13068
验证数据集1452
取出一个样本{'context': '一般来说信用卡预授权期限为30天，预授权发生后30天内，若客户没有进行结算则该预授权将会被取消。 信用卡预授权是特约商户用来作为资金担保的一种手段，在持卡人结束消费一般是办理完酒店退房手续后就可以办理信用卡预授权撤销，信用卡预授权撤销后信用卡额度就会恢复。 当发生预授权交易时，持卡人可以随时通过发卡银行的客服电话或是登录网银来做信用卡预授权查询，信用卡预授权是指发卡机构或其代理机构在特约商户扣款前确认许可冻结额度的交易，预授权会占用卡片的信用额度当客户对预授权进行结算时，该预授权将会被取消，预授权发生后30天内，若客户没有进行结算则该预授权将会被取消。', 'answer': '30天', 'question': '信用卡预授权多久撤销', 'id': 13788}


In [5]:
train_data[0]

{'context': '一般来说信用卡预授权期限为30天，预授权发生后30天内，若客户没有进行结算则该预授权将会被取消。 信用卡预授权是特约商户用来作为资金担保的一种手段，在持卡人结束消费一般是办理完酒店退房手续后就可以办理信用卡预授权撤销，信用卡预授权撤销后信用卡额度就会恢复。 当发生预授权交易时，持卡人可以随时通过发卡银行的客服电话或是登录网银来做信用卡预授权查询，信用卡预授权是指发卡机构或其代理机构在特约商户扣款前确认许可冻结额度的交易，预授权会占用卡片的信用额度当客户对预授权进行结算时，该预授权将会被取消，预授权发生后30天内，若客户没有进行结算则该预授权将会被取消。',
 'answer': '30天',
 'question': '信用卡预授权多久撤销',
 'id': 13788}

In [4]:
def seed_everything(seed=1029):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using {device} device')
seed_everything(42)

Using cuda device


## 加载模型

In [7]:
check_point = "./model/mengzi-t5-base"
model = AutoModelForSeq2SeqLM.from_pretrained(check_point)
tokenizer = AutoTokenizer.from_pretrained(check_point)
model.to(device)

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


T5ForConditionalGeneration(
  (shared): Embedding(32128, 768)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 768)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=768, out_features=768, bias=False)
              (k): Linear(in_features=768, out_features=768, bias=False)
              (v): Linear(in_features=768, out_features=768, bias=False)
              (o): Linear(in_features=768, out_features=768, bias=False)
              (relative_attention_bias): Embedding(32, 12)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseGatedActDense(
              (wi_0): Linear(in_features=768, out_features=2048, bias=False)
              (wi_1): Linear(in_features=768, out_features=2048, bias=False)
              (wo):

In [13]:
def collate_fn(batch_samples):
    batch_inputs, batch_targets = [], []
    for sample in batch_samples:
        question = sample['question']
        context = sample['context']
        answer = sample['answer']
        
        input_text = f"context: {context} question: {question} "
        batch_inputs.append(input_text)
        batch_targets.append(answer)

    batch_data = tokenizer(
        batch_inputs,
        text_target=batch_targets,
        padding=True,
        truncation=True,
        max_length=max_input_length,
        return_tensors="pt"
    )

    # decoder_input_ids = model.prepare_decoder_input_ids_from_labels(labels)
    # end_token_index = torch.where(batch_data['labels'] == tokenizer.eos_token_id)[1]
    return batch_data

In [9]:
train_dataloader = DataLoader(train_data, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
valid_dataloader = DataLoader(valid_data, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

In [10]:
batch = next(iter(train_dataloader))
print(batch.keys())
print('batch shape:', {k: v.shape for k, v in batch.items()})

KeysView({'input_ids': tensor([[    7, 25395,  7368,  ...,     0,     0,     0],
        [    7, 25395,  7368,  ...,     0,     0,     0],
        [    7, 25395,  7368,  ...,  1225,   101,     1],
        ...,
        [    7, 25395,  7368,  ...,     0,     0,     0],
        [    7, 25395,  7368,  ...,     0,     0,     0],
        [    7, 25395,  7368,  ...,     0,     0,     0]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 1, 1, 1],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]]), 'labels': tensor([[    7, 17465,     1,     0,     0,     0],
        [    7,    16,   471,  2964,  1006,     1],
        [    7,  1073,   114,  1026, 12331,     1],
        [ 7973,    50, 14660,    50,     1,     0],
        [    7,  1677,  1892,  1264,     1,     0],
        [    7,   996,  1429,     1,     0,     0],
        [    7,  5536,     1,     0,     0,     0],
   

In [11]:
def train_loop(dataloader, model, optimizer, lr_scheduler, epoch, total_loss):
    progress_bar = tqdm(range(len(dataloader)))
    progress_bar.set_description(f'loss: {0:>7f}')
    finish_batch_num = (epoch-1) * len(dataloader)
    
    model.train()
    for batch, batch_data in enumerate(dataloader, start=1):
        batch_data = batch_data.to(device)
        outputs = model(**batch_data)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        lr_scheduler.step()

        total_loss += loss.item()
        avg_loss = total_loss / (finish_batch_num + batch)

        # 写入 tensorboard
        global_step = finish_batch_num + batch
        if batch % log_step == 0:
            writer.add_scalar('train/avg_loss', avg_loss, global_step)      # 平均loss
            writer.add_scalar('train/batch_loss', loss.item(), global_step) # 瞬时loss
            writer.add_scalar('train/lr', lr_scheduler.get_last_lr()[0], global_step)

        progress_bar.set_description(f'loss: {loss.item():>7f} and avg_loss: {avg_loss:>7f}')
        progress_bar.update(1)
    return total_loss

In [68]:
def test_loop(dataloader, model):
    preds, labels = [], []
    weights_list = [
    (1., 0, 0, 0),               # BLEU-1
    (0.5, 0.5, 0, 0),            # BLEU-2
    (1/3, 1/3, 1/3, 0),          # BLEU-3
    (0.25, 0.25, 0.25, 0.25)     # BLEU-4
    ]
    model.eval()
    for batch_data in tqdm(dataloader):
        batch_data = batch_data.to(device)
        with torch.no_grad():
            generated_tokens = model.generate(
                batch_data["input_ids"],
                attention_mask=batch_data["attention_mask"],
                max_length=max_target_length,
            ).cpu().numpy()
        label_tokens = batch_data["labels"].cpu().numpy()
        
        decoded_preds = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
        label_tokens = np.where(label_tokens != -100, label_tokens, tokenizer.pad_token_id)
        decoded_labels = tokenizer.batch_decode(label_tokens, skip_special_tokens=True)

        preds += [pred.strip() for pred in decoded_preds]
        labels += [[label.strip() for label in decoded_labels]]
        
    smooth = SmoothingFunction().method2
    BLEUs = [ sentence_bleu(labels, preds, w, smooth) for w in weights_list]
    return BLEUs

In [ ]:
optimizer = AdamW(model.parameters(), lr=learning_rate)
lr_scheduler = get_scheduler(
    "cosine",
    optimizer=optimizer,
    num_warmup_steps=0.05*epoch_num*len(train_dataloader),
    num_training_steps=epoch_num*len(train_dataloader),
)

total_loss = 0.
best_bleu = 0.
preds, labels = [], []

for t in range(epoch_num):
    print(f"Epoch {t+1}/{epoch_num}\n-------------------------------")
    total_loss = train_loop(train_dataloader, model, optimizer, lr_scheduler, t+1, total_loss)
    BLEUs = test_loop(valid_dataloader, model)

    print(f"BLEU 1~4 scores: {BLEUs[0]:>5f}, {BLEUs[1]:>5f}, {BLEUs[2]:>5f}, {BLEUs[3]:>5f}")

    if BLEUs[3] > best_bleu:
        best_bleu = BLEUs[3]
        print('Saving new best model...\n')
        torch.save(
            model.state_dict(), 
            f'epoch_{t+1}_valid_bleu_{BLEUs[3]:0.2f}_model_weights.bin'
        )
print("Done!")

Epoch 1/2
-------------------------------


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 182/182 [00:25<00:00,  7.03it/s]


BLEU:50.0, BLEU:0.0, BLEU:0.0, BLEU:0.0
Validation BLEU: 0.00

Epoch 2/2
-------------------------------


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 182/182 [00:27<00:00,  6.53it/s]

BLEU:50.0, BLEU:0.0, BLEU:0.0, BLEU:0.0
Validation BLEU: 0.00

Done!


In [65]:
test_data = QADataset("./dataset/DuReaderQG/train.json")
test, _ = random_split(test_data, [400, len(test_data)-400]) 
test_dataloader = DataLoader(test, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)


model.load_state_dict(
    torch.load('checkpoint_debug.bin', map_location=torch.device('cpu'))
)
tokenizer = AutoTokenizer.from_pretrained("Langboat/mengzi-t5-base")

In [66]:
len(test_dataloader)

50

In [70]:
test_loop(test_dataloader, model)

  0%|          | 0/50 [00:00<?, ?it/s]

[0.73, 0.5934854673873657, 0.47447232297872477, 0.376662783035961]

In [ ]:
weights_list = [
    (1., 0, 0, 0),               # BLEU-1
    (0.5, 0.5, 0, 0),            # BLEU-2
    (1/3, 1/3, 1/3, 0),          # BLEU-3
    (0.25, 0.25, 0.25, 0.25)     # BLEU-4
    ]
model.eval()
for batch_data in tqdm(test_dataloader):
    batch_data = batch_data.to(device)
    with torch.no_grad():
        generated_tokens = model.generate(
            batch_data["input_ids"],
            attention_mask=batch_data["attention_mask"],
            max_length=max_target_length,
        ).cpu().numpy()
    label_tokens = batch_data["labels"].cpu().numpy()
    
    decoded_preds = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
    label_tokens = np.where(label_tokens != -100, label_tokens, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(label_tokens, skip_special_tokens=True)

    preds += [pred.strip() for pred in decoded_preds]
    labels += [[label.strip() for label in decoded_labels]]
    
smooth = SmoothingFunction().method2
BLEUs = [ sentence_bleu(labels, preds, w, smooth) for w in weights_list]
print(f"BLEU 1~4 scores: {BLEUs[0]:>5f}, {BLEUs[1]:>5f}, {BLEUs[2]:>5f}, {BLEUs[3]:>5f}")

  0%|          | 0/13 [00:00<?, ?it/s]

BLEU 1~4 scores: 0.730000, 0.585747, 0.470221, 0.386345


2.5

In [50]:
preds, labels

(['视频播放器',
  '40万元',
  '8岁',
  '香河',
  '24小时',
  '四个',
  '中国原创鞋品牌',
  '答',
  '买超sam',
  '36集',
  '1000元左右',
  '3~10只左右',
  '李秉宪',
  '晚上',
  '10-15分钟左右',
  '兰芝水库系列',
  '480分钟',
  'CorelDRAW',
  '动漫店',
  '两个',
  '1,000字',
  '2560KBps/S',
  '1232mm×787.2mm',
  '约7.5小时',
  '2013年12月4日中午12:00',
  '重庆协和医院',
  '李氏',
  '194*889mm',
  '6周左右',
  'IP68',
  '2号',
  'Prime15烟油',
  '490°C',
  '70左右',
  '419.53°C',
  '4000-6000元',
  '收入的千分之一',
  '7个工作日',
  '两个小时',
  '淳于琼',
  '2017年7月7日',
  '50-60个',
  '无提取比例',
  '几千元',
  '1月31星期五',
  'ACCESS',
  '600元左右',
  '222',
  '爱上3D网',
  '2012',
  '2升',
  '北京466医院',
  '89万美金',
  '深圳康圆矫正',
  '5600以上',
  'SA-1088D',
  '雷公藤多疳片',
  '5月12日晚上20时左右',
  '眉山正健医院',
  '头层皮',
  '两种',
  '几百万至上千万',
  '4个',
  '3个月后',
  '逃税罪',
  '月经周期的第9-10天',
  '马提尼克',
  '64.1公里',
  '2米到2.6米',
  '圆框眼镜',
  '5分钟到2小时',
  '大筒木辉夜',
  '800MHZ',
  '9800',
  '和珅',
  '约306.7公里',
  '1-3天',
  '氯胺酮',
  '三成',
  '十二个小时后',
  '588元',
  'mini',
  '定场诗',
  '2132.63点',
  '2500元',
  '120元',
  '3号线',
  '优优二手车网',


In [ ]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

weights_list = [
    (1., 0, 0, 0),               # BLEU-1
    (0.5, 0.5, 0, 0),            # BLEU-2
    (1/3, 1/3, 1/3, 0),          # BLEU-3
    (0.25, 0.25, 0.25, 0.25)     # BLEU-4
]

smooth = SmoothingFunction().method2
[ sentence_bleu(labels, preds, w, smooth) for w in weights_list], [ sentence_bleu(labels, preds, w) for w in weights_list]

([0.73, 0.5857473858242989, 0.4702212583852878, 0.386344736004831],
 [0.73, 0.5824018536989036, 0.4647354550975047, 0.3792878406932134])